# 02 — NER via LLM

Extract ancient and historical place names (toponyms) from article titles and
abstracts using a locally-running LLM through [Ollama](https://ollama.com),
with structured JSON output constrained by a JSON Schema.

Input: `articles_with_abstract.jsonl` produced by `01_data_collection.ipynb`.
Output: one JSON line per article in `ner_results.jsonl`, each holding the
list of extracted toponyms.

This notebook is checkpoint/resume aware: articles already present in the
output file are skipped on re-run.

## CONFIG

In [ ]:
import json
import logging
from collections import Counter
from pathlib import Path

import jsonlines
import ollama
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("aec_geoparser.ner")

OLLAMA_MODEL = "qwen2.5:32b"
OLLAMA_HOST  = "http://localhost:11434"
INPUT_FILE   = Path("../data/cache/articles_with_abstract.jsonl")
OUTPUT_FILE  = Path("../data/results/ner_results.jsonl")


## JSON schema for structured output

In [ ]:
NER_SCHEMA = {
    "type": "object",
    "properties": {
        "toponyms": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "toponym_raw":        {"type": "string"},
                    "toponym_normalized": {"type": "string"},
                    "entity_type": {
                        "type": "string",
                        "enum": ["site", "settlement", "region", "river",
                                 "mountain", "sanctuary", "necropolis", "other"]
                    },
                    "context_fragment":   {"type": "string"},
                    "confidence": {
                        "type": "string",
                        "enum": ["high", "medium", "low"]
                    }
                },
                "required": ["toponym_raw", "toponym_normalized", "entity_type",
                             "context_fragment", "confidence"]
            }
        }
    },
    "required": ["toponyms"]
}


## Prompts

`SYSTEM_PROMPT` defines the extraction task and the inclusion/exclusion criteria for toponyms; `USER_PROMPT_TEMPLATE` is filled per-article with the title and abstract.

In [ ]:
SYSTEM_PROMPT = """\
You are an expert in ancient Mediterranean and European archaeology.
Your task is to identify ancient and historical place names (toponyms)
in titles and abstracts of archaeological journal articles.

Extract only names that refer to actual geographic locations relevant
to archaeological research. Include:
- Ancient settlement and site names (e.g. Marzabotto/Kainua, Veii,
  Pompeii, Carthage, Alesia)
- Ancient regions or territories (e.g. Etruria, Latium, Magna Graecia,
  Gaul, Hispania)
- Rivers, mountains, or geographic features known from ancient sources
  (e.g. Tiber, Arno, Vesuvius)
- Modern place names that unambiguously refer to a specific ancient
  site (e.g. "Cerveteri" = ancient Caere)

Do NOT include:
- Modern country, region, or city names used in a contemporary
  administrative sense (e.g. "Italy", "France", "Bologna" as a
  modern city)
- Names of institutions, museums, projects, or conferences
- Personal names or ethnic group names (Romans, Etruscans, etc.)
- Generic descriptors ("the site", "the valley", "the region")

For toponym_normalized: use the most common ancient or scholarly form.
For context_fragment: copy the relevant sentence verbatim from the text.
If no qualifying toponyms are present, return an empty "toponyms" array.
"""

USER_PROMPT_TEMPLATE = """\
Extract all ancient and historical place names from this archaeological
article. Use both title and abstract as sources.

Title: {title}
Abstract: {abstract}

Return only valid JSON.
"""


## Ollama client and extraction call

In [ ]:
client = ollama.Client(host=OLLAMA_HOST)


def extract_toponyms(title: str, abstract: str) -> dict:
    """Call the LLM with structured JSON output and return the parsed toponyms payload."""
    response = client.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT_TEMPLATE.format(title=title, abstract=abstract)},
        ],
        format=NER_SCHEMA,
        options={"temperature": 0},
    )
    return json.loads(response["message"]["content"])


## Processing

Resume from `OUTPUT_FILE` if it exists, skip already-processed `article_id`s, and append one result line per article as soon as it is produced.

In [ ]:
def load_processed_ids(path: Path) -> set[int]:
    """Collect article_id values already present in the output file."""
    if not path.exists():
        return set()
    with jsonlines.open(path) as reader:
        return {row["article_id"] for row in reader}


OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

with jsonlines.open(INPUT_FILE) as reader:
    articles = list(reader)

processed_ids = load_processed_ids(OUTPUT_FILE)
remaining = [a for a in articles if a["id"] not in processed_ids]

logger.info("%d articles total, %d already processed, %d remaining",
            len(articles), len(processed_ids), len(remaining))

n_errors = 0
n_toponyms = 0

with jsonlines.open(OUTPUT_FILE, mode="a") as writer:
    for article in tqdm(remaining, desc="NER extraction"):
        article_id = article["id"]
        try:
            payload = extract_toponyms(article.get("title") or "", article.get("abstract") or "")
            toponyms = payload.get("toponyms", [])
            result = {"article_id": article_id, "toponyms": toponyms}
        except Exception as exc:
            logger.warning("Article %s failed: %s", article_id, exc)
            result = {"article_id": article_id, "toponyms": [], "error": str(exc)}
            n_errors += 1
            toponyms = []

        n_toponyms += len(toponyms)
        writer.write(result)


## Final summary

In [ ]:
with jsonlines.open(OUTPUT_FILE) as reader:
    all_results = list(reader)

total_toponyms = sum(len(r.get("toponyms", [])) for r in all_results)
articles_with_hits = sum(1 for r in all_results if r.get("toponyms"))
entity_type_counts = Counter(
    t["entity_type"]
    for r in all_results
    for t in r.get("toponyms", [])
)

print(f"Total processed:        {len(all_results)}")
print(f"Skipped (resumed):      {len(processed_ids)}")
print(f"Errors:                 {sum(1 for r in all_results if 'error' in r)}")
print(f"Total toponyms found:   {total_toponyms}")
print(f"Articles with >=1 hit:  {articles_with_hits}")
print()
print("Breakdown by entity_type:")
for entity_type, count in entity_type_counts.most_common():
    print(f"  {entity_type:<12} {count}")
